In [ ]:
import matplotlib.pyplot as plt
import xarray as xr

from ml_ds.network import LightningModule
from ml_ds.train_CNN import (
    INPUT_FILES,
    INPUT_VARS,
    STATIC_VARS,
    TARGET_VARS,
    initialize_model,
    load_datasets,
)

In [ ]:
def to_xarray(tensor, vars, coords):
    tensor = tensor.detach().numpy()
    return xr.Dataset(
        {
            var: xr.DataArray(tensor[:, var_num, :, :], coords=coords)
            for var_num, var in enumerate(vars)
        }
    )

In [ ]:
# Put test result back into xarray with coordinates.
era5_coords = xr.open_dataset(INPUT_FILES[0])
era5_coords = {
    "valid_time": era5_coords["valid_time"].values[:1],
    "latitude": era5_coords["latitude"].values,
    "longitude": era5_coords["longitude"].values,
}

In [ ]:
train_data, val_data, test_data = load_datasets()
model = initialize_model()

In [ ]:
network = LightningModule.load_from_checkpoint(
    "../lightning_logs/version_1/checkpoints/epoch=0-step=1316.ckpt",
    map_location="cpu",
    model=model,
    train_dataset=train_data,
    val_dataset=val_data,
    test_dataset=test_data,
    batch_size=1, 
    num_workers=1,
)

In [ ]:
test_loader = iter(network.test_dataloader())

In [ ]:
x, y = next(test_loader)
yh = network.forward(x)
x = train_data.input_means + x * train_data.input_sds

In [ ]:
x = to_xarray(x, INPUT_VARS + STATIC_VARS, era5_coords)
y = to_xarray(y, TARGET_VARS, era5_coords)
yh = to_xarray(yh, TARGET_VARS, era5_coords)

In [ ]:
# import matplotlib.colors as mcolors

fig, ax = plt.subplots(1, 3, figsize=(20, 5))

plot_var = "u10"
time_idx = 0

x[plot_var][time_idx].plot.imshow(ax=ax[0])  # , norm=mcolors.Normalize(-15, 15))
y[plot_var][time_idx].plot.imshow(ax=ax[1])  # , norm=mcolors.Normalize(-15, 15))
yh[plot_var][time_idx].plot.imshow(ax=ax[2])  # , norm=mcolors.Normalize(-15, 15))

ax[0].set_title("x: input")
ax[1].set_title("y: truth")
ax[2].set_title("yh: prediction")

In [ ]:
(y[plot_var] - x[plot_var]).sum()

In [ ]:
(yh[plot_var] - x[plot_var]).sum()

In [ ]:
(yh[plot_var] - y[plot_var]).sum()